# ETL Transform: News

This notebook runs the **news ETL pipeline**: ingest from Postgres → transform → save to `financial_news_transformed` → publish to S3.

**Two modes (set `USE_AGENTIC_ONLY` in the next cell):**
- **VADER (default)**: transform with sentiment (VADER), intent, keywords, tickers; then save and S3.
- **Agentic only**: skip VADER; run LLM-based financial metrics extraction only; then save and S3.

**S3 upload modes:**
- **Per-article**: one CSV per article at `news/crypto/[agentic=true|false/]year=.../.../format=csv/{id}.csv`
- **Batch (run)**: one CSV per run at `news/transformed/crypto/.../batch=run/...`
- **Batch (week/month/year)**: under `news/transformed/crypto/year=.../week=...` etc.

Set `AWS_NEWS_BUCKET` or `AWS_DEFAULT_BUCKET` in `.env` for S3 uploads. For agentic mode, set `OPENAI_API_KEY` or `ANTHROPIC_API_KEY` and optionally `LLM_PROVIDER` (default: openai).

In [ ]:
import sys
from pathlib import Path

# Resolve project root: run from repo root or notebooks/etl/
_cwd = Path(".").resolve()
project_root = _cwd if (_cwd / "src").is_dir() else (_cwd.parent.parent if _cwd.name == "etl" else _cwd)
src_path = project_root / "src"
if src_path.is_dir():
    sys.path.insert(0, str(project_root))
    sys.path.insert(0, str(src_path))
else:
    raise FileNotFoundError(f"Expected src at {src_path}. Run from repo root or notebooks/etl/.")

import pandas as pd

In [ ]:
# Options: set to True to skip VADER and use only agentic (LLM) enrichment
USE_AGENTIC_ONLY = False
SINCE = "2025-01-01"
UNTIL = "2025-12-31"
# For agentic-only: limit rows (None = no limit; set e.g. 10 for a quick test)
AGENTIC_MAX_ROWS = None
# S3: per-article and/or batch (same for both modes)
UPLOAD_S3_PER_ARTICLE = True
UPLOAD_S3_BATCH = ["run", "week", "month", "year"]

In [ ]:
# Run news ETL: either agentic-only (skip VADER) or VADER transform; then save to Postgres and S3.

from datetime import datetime
from pipelines.etl_transform import (
    run_news_etl,
    save_transformed_news_to_postgres,
    _ensure_news_transformed_table,
    build_s3_key_news_per_article,
    build_s3_key_news_batch,
    upload_dataframe_to_s3_key,
)
from config.settings import get_settings
from storage.postgres.pgConn import PgConn
from storage.postgres import PostgresSQL_table_queries as q

if USE_AGENTIC_ONLY:
    from pipelines.etl_cli import ingest_news
    from agents.registry import get_llm_client
    from agents.transforms.agentic_transform import AgenticTextEnricher, FinancialMetricsTask
    from storage.cloud.CloudStorage import CloudStorageProvider

    df = ingest_news(since=SINCE, until=UNTIL)
    if df is None or df.empty:
        transformed_df = pd.DataFrame()
        print("No news data to process")
    else:
        settings = get_settings()
        provider = getattr(settings.agent, "provider", "openai")
        try:
            client = get_llm_client(provider)
            enricher = AgenticTextEnricher(client=client, task=FinancialMetricsTask())
            transformed_df = enricher.enrich_dataframe(df, max_rows=AGENTIC_MAX_ROWS)
        except (KeyError, ValueError) as e:
            print(f"Agentic skipped (LLM not configured): {e}")
            transformed_df = pd.DataFrame()

        if not transformed_df.empty:
            conn = PgConn(q.FINANCIAL_NEWS_TABLE_NAME)
            _ensure_news_transformed_table(conn)
            n = save_transformed_news_to_postgres(conn, transformed_df, agentic_enabled=True)
            conn.close_connection()
            print(f"Saved {n} rows to {q.FINANCIAL_NEWS_TRANSFORMED_TABLE_NAME}")

            bucket = getattr(settings.aws, "news_bucket", None) or settings.aws.default_bucket
            if bucket and (UPLOAD_S3_PER_ARTICLE or UPLOAD_S3_BATCH):
                aws = CloudStorageProvider.AWS()
                if UPLOAD_S3_PER_ARTICLE:
                    for _, row in transformed_df.iterrows():
                        aid = str(row.get("id", ""))
                        dt_str = row.get("datetime")
                        try:
                            dt = pd.to_datetime(dt_str) if dt_str else datetime.utcnow()
                        except Exception:
                            dt = datetime.utcnow()
                        key = build_s3_key_news_per_article(aid, dt, agentic=True)
                        upload_dataframe_to_s3_key(aws.s3_client, bucket, key, pd.DataFrame([row]))
                    print(f"Uploaded {len(transformed_df)} per-article CSVs to s3://{bucket}/")
                if UPLOAD_S3_BATCH:
                    now = datetime.utcnow()
                    for batch_type in UPLOAD_S3_BATCH:
                        key = build_s3_key_news_batch(batch_type, now, agentic=True)
                        upload_dataframe_to_s3_key(aws.s3_client, bucket, key, transformed_df)
                        print(f"Uploaded batch {batch_type} to s3://{bucket}/{key}")

    print(f"Agentic: transformed {len(transformed_df)} articles")
else:
    transformed_df = run_news_etl(
        since=SINCE,
        until=UNTIL,
        news_bucket=None,
        save_to_postgres=True,
        upload_s3_per_article=UPLOAD_S3_PER_ARTICLE,
        upload_s3_batch=UPLOAD_S3_BATCH if UPLOAD_S3_BATCH else None,
        sentiment_backend="vader",
        extract_tickers=True,
    )
    print(f"VADER: transformed {len(transformed_df)} articles")

In [ ]:
# Inspect transformed output
if not transformed_df.empty:
    display(transformed_df.head())
    print(transformed_df.columns.tolist())

## Optional: Per-article only (no batch)
Use when you want only one CSV per article in S3.

In [ ]:
# transformed_per_article = run_news_etl(
#     since="2026-01-01",
#     until="2026-01-28",
#     save_to_postgres=True,
#     upload_s3_per_article=True,
#     upload_s3_batch=None,
# )

## Optional: Batch only (no per-article)
Use when you want a single CSV per run, or per week/month/year.

In [ ]:
# transformed_batch = run_news_etl(
#     date="2026-01-27",
#     save_to_postgres=True,
#     upload_s3_per_article=False,
#     upload_s3_batch=["run", "month"],
# )